In [ ]:
import inspect
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gymnasium import spaces
from IPython.display import display

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'configs').exists() and (candidate / 'environment').exists() and (candidate / 'experts').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root.')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import IQN allocator instead of PPO
from allocator.allocator_agent import IQNAllocator
from allocator.allocator_env import AllocatorEnv
from allocator.train_allocator import (
    build_base_env,
    compute_max_drawdown,
    compute_sharpe,
    compute_sortino,
    load_config,
    load_expert_agent,
)
from experts.train_experts import EXPERT_NAME_MAP, detect_asset_type
from regime.config import CONFIG as REGIME_CONFIG
from regime.inference import MultiHorizonRegimeClassifier

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
ASSET = 'EURUSD'
DEVICE = 'cpu'

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'allocator.yaml'
EXPERTS_CONFIG_PATH = PROJECT_ROOT / 'configs' / 'experts.yaml'
TEST_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / ASSET / 'train.csv'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints' / ASSET / 'diversity'
ALLOCATOR_PATH = CHECKPOINT_DIR  / 'allocator.pt'          # <-- this should be the IQN checkpoint
REGIME_MODEL_PATH = CHECKPOINT_DIR.parent / 'regime' / ASSET / f'{ASSET}_tcn_multi.pth'

alloc_cfg = load_config(CONFIG_PATH)
experts_cfg = load_config(EXPERTS_CONFIG_PATH)
features = list(experts_cfg['ppo']['features'])
asset_type = detect_asset_type(ASSET)
lookback = int(alloc_cfg['allocator'].get('lookback', 20))

test_df = pd.read_csv(TEST_DATA_PATH)
base_env_template = build_base_env(test_df, features, asset_type)

expert_names = [EXPERT_NAME_MAP[i] for i in range(1, 7)]
experts = []
for name in expert_names:
    ckpt_path = CHECKPOINT_DIR  / name.upper() / f'{name.lower()}_expert.pt'
    expert = load_expert_agent(name, ckpt_path, base_env_template, device=DEVICE, expert_cfg=experts_cfg)
    experts.append(expert)

# Load regime model (optional)
regime_model = None
if REGIME_MODEL_PATH.exists():
    print(f"Loading regime model from {REGIME_MODEL_PATH}")
    regime_model = MultiHorizonRegimeClassifier(REGIME_MODEL_PATH)
else:
    print(f"Regime model not found at {REGIME_MODEL_PATH}, proceeding without regime features.")

# Create environment with regime model
allocator_env = AllocatorEnv(
    build_base_env(test_df, features, asset_type),
    experts,
    lookback=lookback,
    regime_model=regime_model,
    regime_feature_cols=list(REGIME_CONFIG['features']),
    regime_window=int(REGIME_CONFIG['model']['window']),
)

# --- Instantiate IQN allocator with hyperparameters from config ---
iqn_params = alloc_cfg['allocator']
allocator = IQNAllocator(
    state_dim=allocator_env.observation_space.shape[0],
    action_dim=6,
    lr=iqn_params.get('lr', 3e-4),
    gamma=iqn_params.get('gamma', 0.995),
    buffer_size=iqn_params.get('buffer_size', 100000),
    batch_size=iqn_params.get('batch_size', 64),
    hidden_dim=iqn_params.get('hidden_dim', 256),
    n_quantiles=iqn_params.get('n_quantiles', 32),
    embedding_dim=iqn_params.get('embedding_dim', 64),
    num_layers=iqn_params.get('num_layers', 2),
    activation=iqn_params.get('activation', 'relu'),
    target_update_freq=0,
    epsilon_start=0,
    epsilon_end=0,
    epsilon_decay=0,
    risk_tau=iqn_params.get('risk_tau', 0.5),
    device=DEVICE,
)
allocator.load(ALLOCATOR_PATH)   # load the saved IQN checkpoint

print(f'Asset: {ASSET} ({asset_type})')
print(f'Features: {len(features)}')
print(f'Allocator observation dim: {allocator_env.observation_space.shape[0]}')
print(f'Allocator checkpoint: {ALLOCATOR_PATH}')

In [ ]:
def call_expert_act(expert, obs):
    act_fn = getattr(expert, 'act')
    try:
        signature = inspect.signature(act_fn)
    except (TypeError, ValueError):
        signature = None

    if signature is not None and 'eval_mode' in signature.parameters:
        try:
            raw = act_fn(obs, eval_mode=True)
        except TypeError:
            raw = act_fn(obs)
    else:
        raw = act_fn(obs)

    if isinstance(raw, tuple):
        raw = raw[0]
    return raw


def expert_output_to_signal(action):
    if isinstance(action, np.ndarray):
        flat = action.reshape(-1)
        if flat.size == 0:
            raise ValueError('Expert action array is empty.')
        action = flat[0]

    if isinstance(action, (np.integer, int)):
        mapping = {0: -1.0, 1: 0.0, 2: 1.0}
        discrete = int(action)
        if discrete in mapping:
            return mapping[discrete]
        return float(np.clip(discrete, -1, 1))

    return float(np.clip(float(action), -1.0, 1.0))


def signal_to_base_action(signal, base_env):
    continuous = bool(getattr(base_env, 'continuous_actions', False))
    action_space = getattr(base_env, 'action_space', None)
    if continuous or isinstance(action_space, spaces.Box):
        return np.asarray([signal], dtype=np.float32)
    if signal <= -0.33:
        return 0
    if signal >= 0.33:
        return 2
    return 1


def expert_to_base_action(expert, obs, base_env):
    return signal_to_base_action(expert_output_to_signal(call_expert_act(expert, obs)), base_env)


def run_base_policy(base_env, policy_fn):
    obs, _ = base_env.reset()
    equities = [float(base_env.initial_balance)]
    actions = []
    terminated = False
    truncated = False

    while not (terminated or truncated):
        action = policy_fn(obs)
        obs, reward, terminated, truncated, info = base_env.step(action)
        equities.append(float(info.get('equity', base_env.initial_balance)))
        actions.append(action)

    return np.asarray(equities, dtype=np.float64), np.asarray(actions, dtype=object)


def run_expert(base_env, expert):
    return run_base_policy(base_env, lambda obs: expert_to_base_action(expert, obs, base_env))


def run_uniform_ensemble(base_env, expert_list):
    def policy(obs):
        signals = [expert_output_to_signal(call_expert_act(exp, obs)) for exp in expert_list]
        return np.asarray([float(np.mean(signals))], dtype=np.float32)

    return run_base_policy(base_env, policy)


def run_allocator(allocator_env, allocator_agent):
    obs, _ = allocator_env.reset()
    equities = [float(allocator_env.base_env.initial_balance)]
    actions = []
    terminated = False
    truncated = False

    while not (terminated or truncated):
        action = allocator_agent.act(obs, deterministic=True)
        obs, reward, terminated, truncated, info = allocator_env.step(action)
        equities.append(float(info.get('equity', allocator_env.base_env.initial_balance)))
        actions.append(int(action))

    return np.asarray(equities, dtype=np.float64), np.asarray(actions, dtype=np.int64)


def compute_metrics(equity_curve):
    equity_curve = np.asarray(equity_curve, dtype=np.float64)
    if equity_curve.size < 2:
        return {
            'total_return': 0.0,
            'annual_return': 0.0,
            'annual_vol': 0.0,
            'sharpe': 0.0,
            'sortino': 0.0,
            'max_drawdown': 0.0,
            'win_rate': 0.0,
        }

    returns = np.diff(equity_curve) / np.maximum(equity_curve[:-1], 1e-8)
    total_return = float((equity_curve[-1] - equity_curve[0]) / max(abs(equity_curve[0]), 1e-8))
    ann_vol = float(np.std(returns) * np.sqrt(252)) if returns.size > 1 else 0.0
    growth = max(1.0 + total_return, 1e-8)
    ann_return = float(growth ** (252.0 / max(len(returns), 1)) - 1.0) if len(returns) > 0 else 0.0
    sharpe = float(compute_sharpe(returns.tolist()))
    sortino = float(compute_sortino(returns.tolist()))
    max_dd = float(compute_max_drawdown(equity_curve.tolist()))
    win_rate = float(np.mean(returns > 0.0)) if returns.size else 0.0

    return {
        'total_return': total_return,
        'annual_return': ann_return,
        'annual_vol': ann_vol,
        'sharpe': sharpe,
        'sortino': sortino,
        'max_drawdown': max_dd,
        'win_rate': win_rate,
    }


In [ ]:
allocator_equity, allocator_actions = run_allocator(
    AllocatorEnv(
        build_base_env(test_df, features, asset_type),
        experts,
        lookback=lookback,
        regime_model=regime_model,
        regime_feature_cols=list(REGIME_CONFIG['features']),
        regime_window=int(REGIME_CONFIG['model']['window']),
    ),
    allocator,
)

expert_results = {}
for idx, name in enumerate(expert_names):
    equity, actions = run_expert(build_base_env(test_df, features, asset_type), experts[idx])
    expert_results[name] = {'equity': equity, 'actions': actions}

best_name = max(expert_results, key=lambda key: expert_results[key]['equity'][-1])
ensemble_equity, ensemble_actions = run_uniform_ensemble(build_base_env(test_df, features, asset_type), experts)

results = {
    'Allocator': {'equity': allocator_equity, 'actions': allocator_actions},
    f'Best Expert ({best_name})': expert_results[best_name],
    'Uniform Ensemble': {'equity': ensemble_equity, 'actions': ensemble_actions},
}

metrics_rows = []
for name, result in results.items():
    metrics = compute_metrics(result['equity'])
    metrics_rows.append({'strategy': name, **metrics, 'final_equity': float(result['equity'][-1])})

metrics_df = pd.DataFrame(metrics_rows).set_index('strategy')
display(metrics_df.round(4))

selection_counts = np.bincount(np.asarray(allocator_actions, dtype=np.int64), minlength=6)
selection_df = pd.DataFrame({
    'expert': expert_names,
    'count': selection_counts,
    'frequency': selection_counts / max(int(selection_counts.sum()), 1),
})
display(selection_df)


In [ ]:
plt.figure(figsize=(14, 7))
for name, result in results.items():
    eq = result['equity']
    plt.plot(eq / eq[0] - 1.0, label=name, linewidth=2)

plt.axhline(0.0, color='black', linestyle='--', alpha=0.4)
plt.title(f'Allocator vs Baselines ({ASSET})')
plt.xlabel('Step')
plt.ylabel('Cumulative Return')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.bar(selection_df['expert'], selection_df['frequency'], color='steelblue')
plt.title(f'Allocator Expert Selection Distribution ({ASSET})')
plt.xlabel('Expert')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()
